In [0]:
%run ../src/Bronze/landing_ecomm

In [0]:
%run ../src/Silver/Unified_ecomm

In [0]:
%run ../src/Gold/publish_ecomm

In [0]:
    import pytest
    import pyspark.sql.functions as F
    from pyspark.sql.window import Window
    from pyspark.sql.types import *

-----------------**Question 1**------------

In [0]:
#testCase1
import pytest
import pyspark.sql.functions as F

def test_clean_column_spaces_removes_spaces():
    """
    Validates that clean_column_spaces strips all whitespace characters
    """
    mock_data = [("123", "Jatin", "Bangalore")]
    mock_columns = ["Customer ID", "Customer Name ", "Home Base"]
    
    input_df = spark.createDataFrame(mock_data, schema=mock_columns)
    processed_df = clean_column_spaces(input_df)
    expected_columns = ["CustomerID", "CustomerName", "HomeBase"]
    
    assert processed_df.columns == expected_columns

In [0]:
test_clean_column_spaces_removes_spaces()

In [0]:
#testcase 2
#test to check file path exist or not 
import pytest
from pyspark.sql.utils import AnalysisException

def test_excel_read_file_not_found():
    """Edge Case: File does not exist should raise AnalysisException."""
    good_path = "/Volumes/assessment/bronze/bronze_customer/Customer.xlsx"
    
    exception_caught = None
    
    try:
        excel_read(good_path)
    except AnalysisException as e:
        exception_caught = e
        
    # Assert that the exception was actually caught and is not None
    assert exception_caught is None

In [0]:
test_excel_read_file_not_found()

In [0]:
#test case 3
#test to check file path exist or not 
import pytest
from pyspark.sql.utils import AnalysisException

def test_csv_read_file_not_found():
    """Edge Case: File does not exist should raise AnalysisException."""
    good_path = "/Volumes/assessment/bronze/bronze_product/Products.csv"
    
    exception_caught = None
    
    try:
        csv_read(good_path)
    except AnalysisException as e:
        exception_caught = e
        
    # Assert that the exception was actually caught and is not None
    assert exception_caught is None

In [0]:

test_csv_read_file_not_found()

In [0]:
#test case 4
#test to check file path exist or not 
import pytest
from pyspark.sql.utils import AnalysisException

def test_json_read_file_not_found():
    """Edge Case: File does not exist should raise AnalysisException."""
    good_path = "/Volumes/assessment/bronze/bronze_orders/Orders.json"
    
    exception_caught = None
    
    try:
        json_read(good_path)
    except AnalysisException as e:
        exception_caught = e
        
    # Assert that the exception was actually caught and is not None
    assert exception_caught is None

In [0]:
test_json_read_file_not_found()

In [0]:
#test case 5
import pytest
from pyspark.sql import Row
from unittest.mock import patch

def test_excel_read_column_names_match_expected(spark):
    """
    Scenario: Validates that the output of excel_read contains the exact 
    expected column names specified by the data contract.
    """
    # 1. ARRANGE: Define your expected schema contract list
    expected_columns = ['CustomerID', 'CustomerName',  'email',  'phone',  'address',  'Segment',  'Country',  'City',  'State',  'PostalCode',  'Region']

    # Create a mock DataFrame that simulates what your code should output
    mock_data = [Row(CustomerID="CUST-2026-08X",CustomerName="Jatin Sharma",email="jatin.sharma@example.com",phone="+91-9876543210",address="123, 4th Cross, Indiranagar",Segment="Corporate",Country="India",City="Bangalore",State="Karnataka",PostalCode="560038",Region="APAC")]

    df_mock = spark.createDataFrame(mock_data, schema=expected_columns)

    actual_df = excel_read("/Volumes/assessment/bronze/bronze_customer/Customer.xlsx")
        
    actual_columns = actual_df.columns

    # 3. ASSERT: Compare the lists directly
    # Using sorted() ensures the test doesn't fail just because of column order
    assert sorted(actual_columns) == sorted(expected_columns)

In [0]:
test_excel_read_column_names_match_expected(spark)

In [0]:
#test case 6
import pytest
from pyspark.sql import Row

def test_csv_read_column_names_match_expected(spark):
    """
    Scenario: Validates that the output of excel_read contains the exact 
    expected column names specified by the data contract.
    """
    # 1. ARRANGE: Define your expected schema contract list
    expected_columns = ['ProductID',  'Category',  'Sub-Category',  'ProductName',  'State',  'Priceperproduct']

    # Create a mock DataFrame that simulates what your code should output
    mock_data = [Row(ProductID="PROD_001", Category="Technology", Sub_Category="Software", ProductName="Spark Engine", State="Karnataka", Priceperproduct=1200.50)]
    df_mock = spark.createDataFrame(mock_data, schema=expected_columns)

    actual_df = csv_read("/Volumes/assessment/bronze/bronze_product/Products.csv")
        
    actual_columns = actual_df.columns

    # 3. ASSERT: Compare the lists directly
    # Using sorted() ensures the test doesn't fail just because of column order
    assert sorted(actual_columns) == sorted(expected_columns)

In [0]:
test_csv_read_column_names_match_expected(spark)

In [0]:
#test case 7
import pytest
from pyspark.sql import Row

def test_json_read_column_names_match_expected(spark):
    """
    Scenario: Validates that the output of excel_read contains the exact 
    expected column names specified by the data contract.
    """
    # 1. ARRANGE: Define your expected schema contract list
    expected_columns = ['CustomerID',  'Discount',  'OrderDate',  'OrderID',  'Price',  'ProductID',  'Profit',  'Quantity',  'RowID',  'ShipDate',  'ShipMode']

    # Create a mock DataFrame that simulates what your code should output
    mock_data = [Row(CustomerID="CUST_001", Discount=0.15, OrderDate="2026-06-20", OrderID="ORD_999", Price=450.00, ProductID="PROD_77", Profit=67.50, Quantity=2, RowID="R104", ShipDate="2026-06-22", ShipMode="Express")]

    df_mock = spark.createDataFrame(mock_data, schema=expected_columns)

    actual_df = json_read("/Volumes/assessment/bronze/bronze_orders/Orders.json")
        
    actual_columns = actual_df.columns

    # 3. ASSERT: Compare the lists directly
    # Using sorted() ensures the test doesn't fail just because of column order
    assert sorted(actual_columns) == sorted(expected_columns)

In [0]:
test_json_read_column_names_match_expected(spark)

In [0]:
#test case 8
import pytest
from pyspark.sql import Row
import pyspark.sql.functions as F
from unittest.mock import patch

def test_write_handles_idempotency_and_schema_evolution(spark):
    """
    Validates that csv_write overwrites destination data without duplicates
    and dynamically evolves schemas via 'overwriteSchema = true'.
    """
    target_table_csv = "default.test_bronze_csv_sales"
    target_table_json = "default.test_bronze_json_sales"
    target_table_xlsx = "default.test_bronze_xlsx_sales"
    
    spark.sql(f"DROP TABLE IF EXISTS {target_table_csv}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table_json}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table_xlsx}")



    try:
        initial_mock_data = [Row(OrderID="ORD01", Profit=45.50)]
        df_initial = spark.createDataFrame(initial_mock_data)

        # Mock 'csv_read' to return our initial DataFrame payload
        with patch('__main__.csv_read', return_value=df_initial):
            csv_write(target_table_csv, "dummy_path/initial.csv")

        # Mock 'json_read' to return our initial DataFrame payload
        with patch('__main__.json_read', return_value=df_initial):
            json_write(target_table_json, "dummy_path/initial.json")

        # Mock 'xlsx_read' to return our initial DataFrame payload
        with patch('__main__.excel_read', return_value=df_initial):
            excel_write(target_table_xlsx, "dummy_path/initial.xlsx")

        # Verify initial target capture state
        assert spark.table(target_table_csv).count() == 1
        assert spark.table(target_table_json).count() == 1
        assert spark.table(target_table_xlsx).count() == 1
        
        # Trigger identical re-run task to test basic Idempotency 
        with patch('__main__.csv_read', return_value=df_initial):
            csv_write(target_table_csv, "dummy_path/initial.csv")

        # Trigger identical re-run task to test basic Idempotency 
        with patch('__main__.json_read', return_value=df_initial):
            json_write(target_table_json, "dummy_path/initial.json")

        # Trigger identical re-run task to test basic Idempotency 
        with patch('__main__.excel_read', return_value=df_initial):
            excel_write(target_table_xlsx, "dummy_path/initial.xlsx")
            
        # Volume must remain exactly 1
        assert spark.table(target_table_csv).count() == 1
        assert spark.table(target_table_json).count() == 1
        assert spark.table(target_table_xlsx).count() == 1

        # To test the function is mutable and schema evolution can indeed happen --
        # The new payload completely alters the schema profile by introducing a new column 'Category'
        evolved_mock_data = [Row(OrderID="ORD02", Profit=120.00, Category="Technology")]
        df_evolved = spark.createDataFrame(evolved_mock_data)

        # Trigger writing the mutated data schema structure
        with patch('__main__.csv_read', return_value=df_evolved):
            csv_write(target_table_csv, "dummy_path/evolved.csv")
        with patch('__main__.json_read', return_value=df_evolved):
            json_write(target_table_json, "dummy_path/evolved.json")
        with patch('__main__.excel_read', return_value=df_evolved):
            excel_write(target_table_xlsx, "dummy_path/evolved.xlsx")

        # Fetch resulting Delta structure
        csv_df = spark.table(target_table_csv)
        columns_csv = csv_df.columns
        json_df = spark.table(target_table_json)
        columns_json = json_df.columns
        xlsx_df = spark.table(target_table_xlsx)
        columns_xlsx = xlsx_df.columns

        # --- PHASE 3: Structural Assertions ---
        # 1. Row tracking check (old row should be entirely replaced by the 1 new row)
        assert csv_df.count() == 1
        assert json_df.count() == 1
        assert xlsx_df.count() == 1
        
        # 2. Schema check (The table must now include the new 'Category' column)
        assert "Category" in columns_csv, "Delta table failed to evolve its schema dynamically!"
        assert "Category" in columns_json, "Delta table failed to evolve its schema dynamically!"
        assert "Category" in columns_xlsx, "Delta table failed to evolve its schema dynamically!"
        
        # 3. Data asset integrity verification
        assert csv_df.filter(F.col("OrderID") == "ORD02").first()["Category"] == "Technology"
        assert json_df.filter(F.col("OrderID") == "ORD02").first()["Category"] == "Technology"
        assert xlsx_df.filter(F.col("OrderID") == "ORD02").first()["Category"] == "Technology"

    finally:
        # Clean up global workspace metadata artifacts post-execution
        spark.sql(f"DROP TABLE IF EXISTS {target_table_csv}")
        spark.sql(f"DROP TABLE IF EXISTS {target_table_json}")
        spark.sql(f"DROP TABLE IF EXISTS {target_table_xlsx}")

In [0]:
test_write_handles_idempotency_and_schema_evolution(spark)

In [0]:
customer_schema=StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("email", StringType(), True),
        StructField("Segment", StringType(), True),
        StructField("Country", StringType(), True),
        StructField("PostalCode", StringType(), True),
        StructField("phone", StringType(), True)
    ])

In [0]:
#testCase9

def test_dqc_customer_pristine_record_accepted():
    """
    Ensures a perfectly valid customer row passes all checks,
    landing in accepted_df with no errors.
    """

    mock_data = [
        ("CUST001", "Jatin Sharma", "jatin@example.com", "Consumer", "India", "560001", "9876543210")
    ]
    input_df = spark.createDataFrame(mock_data, schema=customer_schema)
    input_df.createOrReplaceTempView("tmp_mock_customer_good")
    accepted_df, rejected_df = dqc_checks_customer("tmp_mock_customer_good")

    assert accepted_df.count() == 1
    assert rejected_df.count() == 0
    
    assert "dq_errors" not in accepted_df.columns

In [0]:
test_dqc_customer_pristine_record_accepted()

In [0]:
#testCase10

def test_dqc_customer_invalid_records_rejected(customer_schema):
    """
    Verifies that rows containing anomalies are routed to rejected_df 
    """
    dirty_data = [
        (None, "Sanjana", "sanjana@test.com", "InvalidSegment", "India", None, "1234567890"),
        ("CUST002", "Sehaj123", "sehaj@test.com", "Corporate", "India", "560002", "1234567890"),
        ("CUST003", "Amit", "amit@test.co", "Home Office", "India", "560003", "#REF! Error")
    ]
    input_df = spark.createDataFrame(dirty_data, schema=customer_schema)
    input_df.createOrReplaceTempView("tmp_mock_customer_dirty")

    accepted_df, rejected_df = dqc_checks_customer("tmp_mock_customer_dirty")

    # Assert: Verify routing isolation split maps out cleanly
    assert accepted_df.count() == 0
    assert rejected_df.count() == 3

    rejected_records = rejected_df.collect()

    # Inspect Row A Flags
    row_a_errors = rejected_records[0]["dq_errors"]
    assert "MISSING_CUSTOMER_ID" in row_a_errors
    assert "INVALID_SEGMENT" in row_a_errors
    assert "MISSING_POSTAL_CODE" in row_a_errors

    # Inspect Row B Flags
    row_b_errors = rejected_records[1]["dq_errors"]
    assert "NAME_CONTAINS_NUMBERS" in row_b_errors

    # Inspect Row C Flags
    row_c_errors = rejected_records[2]["dq_errors"]
    assert "PHONE_HAS_SYSTEM_ERROR" in row_c_errors
    assert "PHONE_INVALID_DIGIT_COUNT" in row_c_errors

In [0]:
test_dqc_customer_invalid_records_rejected(customer_schema)

In [0]:
#testCase11

def test_dqc_customer_duplicate_ids_rejected(customer_schema):
    """
    Tests that the Window.partitionBy function correctly catches and 
    flags multiple non-unique duplicate CustomerIDs.
    """
    duplicate_data = [
        ("CUST_DUP", "Alex", "alex1@test.com", "Consumer", "US", "10001", "1234567890"),
        ("CUST_DUP", "Alex", "alex2@test.com", "Consumer", "US", "10001", "0987654321")
    ]
    input_df = spark.createDataFrame(duplicate_data, schema=customer_schema)
    input_df.createOrReplaceTempView("tmp_mock_customer_dups")

    _, rejected_df = dqc_checks_customer("tmp_mock_customer_dups")

    assert rejected_df.count() == 2
    
    for row in rejected_df.collect():
        assert "DUPLICATE_CUSTOMER_ID" in row["dq_errors"]

In [0]:
test_dqc_customer_duplicate_ids_rejected(customer_schema)

In [0]:
order_schema=StructType([
        StructField("RowID", IntegerType(), True),
        StructField("OrderID", StringType(), True),
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True),
        StructField("OrderDate", StringType(), True),
        StructField("ShipDate", StringType(), True),
        StructField("Quantity", IntegerType(), True),
        StructField("Price", DoubleType(), True),
        StructField("Discount", DoubleType(), True),
        StructField("Profit", DoubleType(), True),
        StructField("ShipMode", StringType(), True)
    ])

In [0]:
#testcase 12
def test_dqc_orders_pristine_record_accepted(order_schema):
    """
    Ensures that a flawless row with correct string-date formats ('d/M/yyyy')
    is accepted cleanly
    """
    good_data = [
        (1, "ORD-2014-001", "CUST-101", "PROD-OFF-01", "30/4/2014", "2/5/2014", 3, 150.00, 0.0, 45.00, "Second Class")
    ]
    input_df = spark.createDataFrame(good_data, schema=order_schema)
    input_df.createOrReplaceTempView("tmp_mock_orders_clean")

    accepted_df, rejected_df = dqc_checks_orders("tmp_mock_orders_clean")

    assert accepted_df.count() == 1
    assert rejected_df.count() == 0

    assert "dq_errors" not in accepted_df.columns

In [0]:
test_dqc_orders_pristine_record_accepted(order_schema)

In [0]:
#testCase 13
def test_dqc_orders_date_anomalies_rejected(order_schema):
    """
    Validates both sides of date-safety checking:
    Row A: Identifies a logical error when OrderDate occurs after ShipDate.
    Row B: Identifies a malformed structure when date format string parsing fails.
    """
    faulty_date_data = [
        (2, "ORD-002", "CUST-102", "PROD-TEC-02", "15/5/2014", "2/5/2014", 1, 10.00, 0.0, 2.00, "Standard Class"),
        (3, "ORD-003", "CUST-103", "PROD-TEC-03", "2014-04-30", "2/5/2014", 1, 20.00, 0.0, 5.00, "Standard Class")
    ]
    input_df = spark.createDataFrame(faulty_date_data, schema=order_schema)
    input_df.createOrReplaceTempView("tmp_mock_orders_bad_dates")

    accepted_df, rejected_df = dqc_checks_orders("tmp_mock_orders_bad_dates")

    assert accepted_df.count() == 0
    assert rejected_df.count() == 2

    rejected_rows = rejected_df.orderBy("RowID").collect()

    assert "LOGICAL_ERROR_ORDER_DATE_AFTER_SHIP_DATE" in rejected_rows[0]["dq_errors"]

    assert "MALFORMED_ORDER_DATE_FORMAT" in rejected_rows[1]["dq_errors"]

In [0]:
test_dqc_orders_date_anomalies_rejected(order_schema)

In [0]:
#testCase 14
def test_dqc_orders_duplicate_row_ids_rejected(order_schema):
    """
    Ensures that Window.partitionBy detects matching RowID records 
    and applies a 'DUPLICATE_ROW_ID' flag across both conflicting instances.
    """

    duplicate_rows = [
        (999, "ORD-004", "CUST-104", "PROD-FUR-01", "01/5/2014", "05/5/2014", 2, 500.00, 0.1, 50.00, "First Class"),
        (999, "ORD-005", "CUST-104", "PROD-FUR-01", "01/5/2014", "05/5/2014", 2, 500.00, 0.1, 50.00, "First Class")
    ]
    input_df = spark.createDataFrame(duplicate_rows, schema=order_schema)
    input_df.createOrReplaceTempView("tmp_mock_orders_dups")

    accepted_df, rejected_df = dqc_checks_orders("tmp_mock_orders_dups")

    assert accepted_df.count() == 0
    assert rejected_df.count() == 2

    for row in rejected_df.collect():
        assert "DUPLICATE_ROW_ID" in row["dq_errors"]

In [0]:
test_dqc_orders_duplicate_row_ids_rejected(order_schema)

In [0]:
product_schema=StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True),
        StructField("Sub-Category", StringType(), True), # Intentionally left dirty with hyphen
        StructField("ProductName", StringType(), True),
        StructField("State", StringType(), True),
        StructField("Priceperproduct", StringType(), True) # Kept as string to test regex parsing
    ])

In [0]:
#testCase 15
def test_dqc_product_clean_record_accepted(product_schema):
    """
    Ensures that a valid record has its columns stripped of spaces/hyphens
    ('Sub-Category' -> 'SubCategory'), passes validations, and lands in accepted_df.
    """
    good_data = [
        ("PROD123", "Technology", "Phones", "iPhone 15", "Karnataka", "79999.00")
    ]
    input_df = spark.createDataFrame(good_data, schema=product_schema)
    input_df.createOrReplaceTempView("tmp_mock_products_clean")

    accepted_df, rejected_df = dqc_checks_product("tmp_mock_products_clean")

    assert accepted_df.count() == 1
    assert rejected_df.count() == 0
    
    assert "SubCategory" in accepted_df.columns
    assert "Sub-Category" not in accepted_df.columns
    

In [0]:
test_dqc_product_clean_record_accepted(product_schema)

In [0]:
#testCase16
def test_dqc_product_invalid_records_rejected(product_schema):
    """
    Validates that text data anomalies (like numbers in State names) are identified,
    routed to rejected_df, and combined into a clean comma-separated string column.
    """
    dirty_data = [
        ("PROD456", "Furniture", "Chairs", "Office Chair", "Karnat4k4!", "1500.00")
    ]
    input_df = spark.createDataFrame(dirty_data, schema=product_schema)
    input_df.createOrReplaceTempView("tmp_mock_products_dirty")

    accepted_df, rejected_df = dqc_checks_product("tmp_mock_products_dirty")

    assert accepted_df.count() == 0
    assert rejected_df.count() == 1

    rejected_row = rejected_df.first()
    expected_error_string = "STATE_CONTAINS_NUMBERS,STATE_CONTAINS_SPECIAL_CHARACTERS"
    assert rejected_row["dq_errors"] == expected_error_string

In [0]:
test_dqc_product_invalid_records_rejected(product_schema)

-----------------**Question 2**------------``

In [0]:
#testCase17
def test_create_customer_product_bridge_execution():
    """
    Validates that create_customer_product deduplicates
    removes the customer 'state' field, and executes clean inner joins.
    """
    order_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True),
        StructField("Quantity", IntegerType(), True)
    ])
    mock_orders = [
        ("CUST01", "PROD01", 2),
        ("CUST01", "PROD01", 5), 
        ("CUST02", "PROD02", 1)
    ]
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("state", StringType(), True) # Target for dropping
    ])
    mock_customers = [
        ("CUST01", "Jatin Sharma", "Karnataka"),
        ("CUST02", "Sanjana", "Haryana")
    ]
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True)
    ])
    mock_products = [
        ("PROD01", "Technology"),
        ("PROD02", "Furniture")
    ]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_test_orders")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_test_customers")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_test_products")

    enriched_df = create_customer_product("v_test_orders", "v_test_customers", "v_test_products")


    assert enriched_df.count() == 2

    columns = enriched_df.columns
    assert "state" not in columns, "The customer 'state' column was not dropped!"
    assert "CustomerName" in columns
    assert "Category" in columns

    cust_1_record = enriched_df.filter(enriched_df.CustomerID == "CUST01").first()
    assert cust_1_record["CustomerName"] == "Jatin Sharma"
    assert cust_1_record["Category"] == "Technology"

In [0]:
test_create_customer_product_bridge_execution()

In [0]:
#testCase18
def test_create_customer_product_with_empty_source_tables():
    """
    Scenario: Validates that if upstream source datasets are empty,
    the function handles it gracefully, returning a 0-row schema-compliant DataFrame.
    """
    # 1. Arrange: Define schemas but pass 0 records
    order_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True)
    ])
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("state", StringType(), True)
    ])
    prod_schema = StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True)
    ])

    spark.createDataFrame([], schema=order_schema).createOrReplaceTempView("v_empty_orders")
    spark.createDataFrame([], schema=cust_schema).createOrReplaceTempView("v_empty_customers")
    spark.createDataFrame([], schema=prod_schema).createOrReplaceTempView("v_empty_products")

    # 2. Act
    result_df = create_customer_product("v_empty_orders", "v_empty_customers", "v_empty_products")

    # 3. Assert
    assert result_df.count() == 0, "Function should handle empty inputs without crashing or blowing up row metrics!"
    assert "state" not in result_df.columns, "Even on empty data, 'state' must be dropped from the schema metadata."

In [0]:
test_create_customer_product_with_empty_source_tables()

In [0]:
#testCase19
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
def test_create_customer_product_deduplicates_bridge_keys():
    """
    Scenario: Validates that multiple identical orders for the same product
    by the same customer are compressed into a single unique bridge record.
    """
    # 1. Arrange: Create highly repetitive order records
    order_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True)
    ])
    # Customer 1 ordered Product A three separate times
    mock_orders = [
        ("CUST_001", "PROD_AAA"),
        ("CUST_001", "PROD_AAA"),
        ("CUST_001", "PROD_AAA")
    ]
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("state", StringType(), True)
    ])
    mock_customers = [("CUST_001", "Jatin Sharma", "Karnataka")]
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True)
    ])
    mock_products = [("PROD_AAA", "Technology")]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_dup_orders")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_dup_customers")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_dup_products")

    # 2. Act
    result_df = create_customer_product("v_dup_orders", "v_dup_customers", "v_dup_products")

    # 3. Assert: Even though there were 3 raw orders, the output must only have 1 row
    assert result_df.count() == 1, "The .distinct() bridge layer failed to filter out duplicate transactions!"

In [0]:
test_create_customer_product_deduplicates_bridge_keys()

-----------------**Question 3**------------``

In [0]:
#testCase 20
def test_create_enriched_orders_left_joins_and_drops_state():
    """
    Validates that create_enriched_orders enriches transactions with customer/product data,
    retains unmatched transactions using left joins, and drops the customer 'state' field.
    """
    order_schema = StructType([
        StructField("OrderID", StringType(), True),
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True),
        StructField("Profit", DoubleType(), True)
    ])
    mock_orders = [
        ("ORD_01", "CUST_01", "PROD_01", 120.50),
        ("ORD_02", "CUST_02", "MISSING_PROD", 45.00) 
    ]
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("state", StringType(), True)  # Targeted for dropping
    ])
    mock_customers = [
        ("CUST_01", "Jatin Sharma", "Karnataka"),
        ("CUST_02", "Sanjana", "Haryana")
    ]
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True)
    ])
    mock_products = [
        ("PROD_01", "Technology")
    ]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_orders_raw")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_customers_raw")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_products_raw")

    enriched_df = create_enriched_orders("v_orders_raw", "v_customers_raw", "v_products_raw")

    assert enriched_df.count() == 2

    output_columns = enriched_df.columns
    assert "state" not in output_columns
    assert "CustomerName" in output_columns
    assert "Category" in output_columns

    matched_row = enriched_df.filter(F.col("OrderID") == "ORD_01").first()
    assert matched_row["CustomerName"] == "Jatin Sharma"
    assert matched_row["Category"] == "Technology"

    unmatched_row = enriched_df.filter(F.col("OrderID") == "ORD_02").first()
    assert unmatched_row["CustomerName"] == "Sanjana"
    assert unmatched_row["Category"] is None, "The Category field should fall back to None for unmatched left-join fields."

In [0]:
test_create_enriched_orders_left_joins_and_drops_state()

%md
-----------------**Question 3a**------------``

In [0]:
#testCase 21
#test for Profit rounded to 2 decimal places
def test_create_enriched_sales_transforms_and_selects_columns():
    """
    Validates that create_enriched_sales performs correct left joins,
    rounds profit metrics to 2 decimal places
    """
    order_schema = StructType([
        StructField("RowID", IntegerType(), True),
        StructField("OrderID", StringType(), True),
        StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True),
        StructField("OrderDate", StringType(), True),
        StructField("ShipDate", StringType(), True),
        StructField("Quantity", IntegerType(), True),
        StructField("Price", DoubleType(), True),
        StructField("Discount", DoubleType(), True),
        StructField("Profit", DoubleType(), True), # Testing rounding constraint
        StructField("ShipMode", StringType(), True)
    ])

    mock_orders = [
        (1, "ORD_01", "C_01", "P_01", "30/4/2014", "2/5/2014", 3, 100.0, 0.0, 45.126, "Standard Class"),
        (2, "ORD_02", "C_02", "MISSING_P", "01/5/2014", "05/5/2014", 1, 10.0, 0.0, -5.000, "Same Day")
    ]
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("Country", StringType(), True)
    ])
    mock_customers = [
        ("C_01", "Jatin Sharma", "India"),
        ("C_02", "Sanjana", "India")
    ]
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True),
        StructField("Category", StringType(), True),
        StructField("SubCategory", StringType(), True)
    ])
    mock_products = [
        ("P_01", "Technology", "Phones")
    ]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_silver_orders")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_silver_customers")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_silver_products")


    result_df = create_enriched_sales(
        order_table="v_silver_orders", 
        cust_table="v_silver_customers", 
        prod_table="v_silver_products",
        target_table="dummy_gold_path"
    )


    assert result_df.count() == 2


    records = result_df.orderBy("RowID").collect()
    assert records[0]["Profit"] == 45.13, "Floating-point profit did not round up to 2 decimals properly!"
    assert records[1]["Profit"] == -5.00


    assert records[0]["Category"] == "Technology"
    assert records[1]["Category"] is None


    expected_column_sequence = [
        "OrderID", "OrderDate", "RowID", "Quantity", "Price", "Discount", 
        "ShipDate", "ShipMode", "Profit", "CustomerName", "Country", 
        "Category", "SubCategory", "CustomerID", "ProductID"
    ]
    assert result_df.columns == expected_column_sequence, "The output column sequence or names do not match the BI spec!"

In [0]:
test_create_enriched_sales_transforms_and_selects_columns()

%md
-----------------**Question 3b**------------``

In [0]:
#testCase 22
# Create an enriched table which has
# Customer name and country

import pytest
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

def test_create_enriched_sales_customer_country_dimension(spark):
    """
    1. Validates successful enrichment of CustomerName and Country.
    2. Edge Case: Asserts that an orphaned CustomerID doesn't drop the order 
       and handles missing dimension attributes gracefully with NULL padding.
    """
    # Arrange: Match the full output projection column requirements
    order_schema = StructType([
        StructField("OrderID", StringType(), True), StructField("OrderDate", StringType(), True),
        StructField("RowID", StringType(), True), StructField("Quantity", IntegerType(), True),
        StructField("Price", DoubleType(), True), StructField("Discount", DoubleType(), True),
        StructField("ShipDate", StringType(), True), StructField("ShipMode", StringType(), True),
        StructField("Profit", DoubleType(), True), StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True)
    ])
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True), 
        StructField("CustomerName", StringType(), True),
        StructField("Country", StringType(), True)
    ])
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True), 
        StructField("Category", StringType(), True), 
        StructField("SubCategory", StringType(), True) # Matches function .select()
    ])

    # 2 Orders: One happy path, one with a missing customer ID to test left join padding
    mock_orders = [
        Row(OrderID="ORD_01", OrderDate="2026-06-20", RowID="R1", Quantity=1, Price=100.0, 
            Discount=0.0, ShipDate="2026-06-22", ShipMode="Standard", Profit=15.879, 
            CustomerID="CUST_01", ProductID="PROD_01"),
        Row(OrderID="ORD_02", OrderDate="2026-06-20", RowID="R2", Quantity=2, Price=50.0, 
            Discount=0.0, ShipDate="2026-06-22", ShipMode="Express", Profit=5.454, 
            CustomerID="MISSING_CUST", ProductID="PROD_01")
    ]
    
    mock_customers = [Row(CustomerID="CUST_01", CustomerName="Jatin Sharma", Country="India")]
    mock_products = [Row(ProductID="PROD_01", Category="Technology", SubCategory="Software")]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_cust_orders")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_cust_customers")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_cust_products")

    # Act
    result_df = create_enriched_sales("v_cust_orders", "v_cust_customers", "v_cust_products", "dummy_target")
    results = {row["OrderID"]: row for row in result_df.collect()}

    # Assert
    assert result_df.count() == 2, "Left join shouldn't drop core order data lines."
    
    # Happy Path Validation
    assert results["ORD_01"]["CustomerName"] == "Jatin Sharma"
    assert results["ORD_01"]["Country"] == "India"
    
    # Left Join Null Padding Verification
    assert results["ORD_02"]["CustomerName"] is None
    assert results["ORD_02"]["Country"] is None

In [0]:
test_create_enriched_sales_customer_country_dimension(spark)

%md
-----------------**Question 3c**------------``

In [0]:
#testCase 23
# Create an enriched table which has
# Product category and sub category

def test_create_enriched_sales_productCat_subCat_dimension(spark):
    """
    1. Validates successful enrichment of Category and SubCategory columns.
    2. Edge Case: Verifies product catalogs with blank parameters resolve 
       without crashing downstream projection schemas.
    """
    # Arrange
    order_schema = StructType([
        StructField("OrderID", StringType(), True), StructField("OrderDate", StringType(), True),
        StructField("RowID", StringType(), True), StructField("Quantity", IntegerType(), True),
        StructField("Price", DoubleType(), True), StructField("Discount", DoubleType(), True),
        StructField("ShipDate", StringType(), True), StructField("ShipMode", StringType(), True),
        StructField("Profit", DoubleType(), True), StructField("CustomerID", StringType(), True),
        StructField("ProductID", StringType(), True)
    ])
    
    cust_schema = StructType([
        StructField("CustomerID", StringType(), True), StructField("CustomerName", StringType(), True),
        StructField("Country", StringType(), True)
    ])
    
    prod_schema = StructType([
        StructField("ProductID", StringType(), True), 
        StructField("Category", StringType(), True), 
        StructField("SubCategory", StringType(), True) # Un-hyphened to pass function constraints
    ])

    # 2 Orders: One clean product reference, one dirty product reference
    mock_orders = [
        Row(OrderID="ORD_01", OrderDate="2026-06-20", RowID="R1", Quantity=1, Price=10.0, 
            Discount=0.0, ShipDate="2026-06-22", ShipMode="Standard", Profit=2.50, 
            CustomerID="CUST_01", ProductID="PROD_CLEAN"),
        Row(OrderID="ORD_02", OrderDate="2026-06-20", RowID="R2", Quantity=1, Price=20.0, 
            Discount=0.0, ShipDate="2026-06-22", ShipMode="Standard", Profit=4.00, 
            CustomerID="CUST_01", ProductID="PROD_DIRTY")
    ]
    
    mock_customers = [Row(CustomerID="CUST_01", CustomerName="Sanjana", Country="India")]
    
    mock_products = [
        Row(ProductID="PROD_CLEAN", Category="Furniture", SubCategory="Chairs"),
        Row(ProductID="PROD_DIRTY", Category="", SubCategory=None) # Empty string and explicit NULL entry
    ]

    spark.createDataFrame(mock_orders, schema=order_schema).createOrReplaceTempView("v_prod_orders")
    spark.createDataFrame(mock_customers, schema=cust_schema).createOrReplaceTempView("v_prod_customers")
    spark.createDataFrame(mock_products, schema=prod_schema).createOrReplaceTempView("v_prod_products")

    # Act
    result_df = create_enriched_sales("v_prod_orders", "v_prod_customers", "v_prod_products", "dummy_target")
    results = {row["OrderID"]: row for row in result_df.collect()}

    # Assert
    assert "Category" in result_df.columns
    assert "SubCategory" in result_df.columns
    
    # Happy Path Validation
    assert results["ORD_01"]["Category"] == "Furniture"
    assert results["ORD_01"]["SubCategory"] == "Chairs"
    
    # Missing/Blank Metadata Propagation Check
    assert results["ORD_02"]["Category"] == ""
    assert results["ORD_02"]["SubCategory"] is None

In [0]:
test_create_enriched_sales_productCat_subCat_dimension(spark)

%md
-----------------**Question 4**------------``

In [0]:
#testCase 24
# Create an aggregate table that shows profit by 
# Year
# Customer


def test_create_gold_profit_aggregation_calculates_and_sorts_metrics():
    """
    Validates that create_gold_profit_aggregation extracts the correct year from strings,
    aggregates sums and rounds profits
    """

    enriched_schema = StructType([
        StructField("OrderDate", StringType(), True),
        StructField("Category", StringType(), True),
        StructField("SubCategory", StringType(), True),
        StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True),
        StructField("Profit", DoubleType(), True)
    ])


    mock_data = [
        ("21/8/2016", "Technology", "Phones", "C_01", "Jatin Sharma", 100.50),
        ("25/8/2016", "Technology", "Phones", "C_01", "Jatin Sharma", 50.05),
        ("02/11/2017", "Technology", "Phones", "C_01", "Jatin Sharma", 200.00),
        ("15/9/2016", "Technology", "Phones", "C_02", "Sanjana", 300.00)
    ]

    input_table_name = "tmp_mock_gold_enriched"
    spark.createDataFrame(mock_data, schema=enriched_schema).createOrReplaceTempView(input_table_name)

    aggregated_df = create_gold_profit_aggregation(input_table_name)


    results = aggregated_df.collect()
    assert len(results) == 3

    expected_columns = ["Year", "Category", "SubCategory", "CustomerID", "CustomerName", "TotalProfit"]
    assert aggregated_df.columns == expected_columns

    assert results[0]["Year"] == 2016
    assert results[0]["CustomerName"] == "Sanjana"
    assert results[0]["TotalProfit"] == 300.00

 
    assert results[1]["Year"] == 2016
    assert results[1]["CustomerName"] == "Jatin Sharma"
    assert results[1]["TotalProfit"] == 150.55  # 100.50 + 50.05

    assert results[2]["Year"] == 2017
    assert results[2]["TotalProfit"] == 200.00

In [0]:
test_create_gold_profit_aggregation_calculates_and_sorts_metrics()

%md
-----------------**Question 4a**------------``

In [0]:
#testCase 25
# Create an aggregate table that shows profit by 
# Year
import pytest
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import pyspark.sql.functions as F

def test_aggregation_by_year_and_date_parsing(spark):
    """
    Usecase 1: Verifies profit aggregates correctly across different years.
    Edge Case: Tests that the 'd/M/yyyy' string format correctly parses varied lengths (e.g. 5/6 vs 25/12).
    """
    schema = StructType([
        StructField("OrderDate", StringType(), True), StructField("Category", StringType(), True),
        StructField("SubCategory", StringType(), True), StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True), StructField("Profit", DoubleType(), True)
    ])

    # 3 orders: Two in 2026 (varying date lengths), one in 2025. Rest of the dimensions stay identical.
    mock_data = [
        Row(OrderDate="5/6/2026", Category="Tech", SubCategory="Dev", CustomerID="C1", CustomerName="Jatin", Profit=100.50),
        Row(OrderDate="25/12/2026", Category="Tech", SubCategory="Dev", CustomerID="C1", CustomerName="Jatin", Profit=50.25),
        Row(OrderDate="01/01/2025", Category="Tech", SubCategory="Dev", CustomerID="C1", CustomerName="Jatin", Profit=75.00)
    ]
    
    spark.createDataFrame(mock_data, schema=schema).createOrReplaceTempView("v_year_orders")

    # Act
    result_df = create_gold_profit_aggregation("v_year_orders")
    results = result_df.collect()

    # Assert
    # We expect exactly two rows back: One for 2026 group, one for 2025 group
    assert result_df.count() == 2
    
    res_map = {row["Year"]: row["TotalProfit"] for row in results}
    assert res_map[2026] == 150.75, "Math failed or date parsing issue for 2026!"
    assert res_map[2025] == 75.00

In [0]:
test_aggregation_by_year_and_date_parsing(spark)

%md
-----------------**Question 4b**------------``

In [0]:
#testCase 26
# Create an aggregate table that shows profit by 
# Product Category
def test_aggregation_by_product_category_and_blank_strings(spark):
    """
    Usecase 2: Verifies profit aggregates correctly by Product Category.
    Edge Case: Tests resiliency when a Category name is an empty string "".
    """
    schema = StructType([
        StructField("OrderDate", StringType(), True), StructField("Category", StringType(), True),
        StructField("SubCategory", StringType(), True), StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True), StructField("Profit", DoubleType(), True)
    ])

    # 3 orders: 2 in 'Technology', 1 in an empty string category ''
    mock_data = [
        Row(OrderDate="20/06/2026", Category="Technology", SubCategory="Cloud", CustomerID="C1", CustomerName="Sanjana", Profit=500.00),
        Row(OrderDate="20/06/2026", Category="Technology", SubCategory="Cloud", CustomerID="C1", CustomerName="Sanjana", Profit=250.123),
        Row(OrderDate="20/06/2026", Category="",           SubCategory="Cloud", CustomerID="C1", CustomerName="Sanjana", Profit=10.00)
    ]
    
    spark.createDataFrame(mock_data, schema=schema).createOrReplaceTempView("v_cat_orders")

    # Act
    result_df = create_gold_profit_aggregation("v_cat_orders")
    results = result_df.collect()

    # Assert
    assert result_df.count() == 2
    
    res_map = {row["Category"]: row["TotalProfit"] for row in results}
    assert res_map["Technology"] == 750.12 # Handles 2 decimal place profit as well
    assert res_map[""] == 10.00, "Empty string category group was corrupted or lost!"

In [0]:
test_aggregation_by_product_category_and_blank_strings(spark)

%md
-----------------**Question 4c**------------``

In [0]:
#testCase 27
# Create an aggregate table that shows profit by 
# Product Sub Category
def test_aggregation_by_sub_category_with_negative_profits(spark):
    """
    Usecase 3: Verifies profit aggregates correctly by Product Sub-Category.
    Edge Case: Asserts that negative profits (losses) subtract cleanly from the totals.
    """
    schema = StructType([
        StructField("OrderDate", StringType(), True), StructField("Category", StringType(), True),
        StructField("SubCategory", StringType(), True), StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True), StructField("Profit", DoubleType(), True)
    ])

    # SubCategory 'Chairs' has a profit and a loss. SubCategory 'Phones' is completely independent.
    mock_data = [
        Row(OrderDate="20/06/2026", Category="Furniture", SubCategory="Chairs", CustomerID="C1", CustomerName="User", Profit=150.00),
        Row(OrderDate="20/06/2026", Category="Furniture", SubCategory="Chairs", CustomerID="C1", CustomerName="User", Profit=-50.50), # Net should be 99.50
        Row(OrderDate="20/06/2026", Category="Furniture", SubCategory="Phones", CustomerID="C1", CustomerName="User", Profit=300.00)
    ]
    
    spark.createDataFrame(mock_data, schema=schema).createOrReplaceTempView("v_subcat_orders")

    # Act
    result_df = create_gold_profit_aggregation("v_subcat_orders")
    results = result_df.collect()

    # Assert
    assert result_df.count() == 2
    
    res_map = {row["SubCategory"]: row["TotalProfit"] for row in results}
    assert res_map["Chairs"] == 99.50, f"Negative profit math failed! Found: {res_map['Chairs']}"
    assert res_map["Phones"] == 300.00

In [0]:
test_aggregation_by_sub_category_with_negative_profits(spark)

%md
-----------------**Question 4d**------------``

In [0]:
#testCase 28
# Create an aggregate table that shows profit by 
# Customer
def test_aggregation_by_customer_and_ranking_order(spark):
    """
    Usecase 4: Verifies profit aggregates correctly by Customer.
    Edge Case: Asserts that final rows are sorted dynamically by TotalProfit in DESCENDING order.
    """
    schema = StructType([
        StructField("OrderDate", StringType(), True), StructField("Category", StringType(), True),
        StructField("SubCategory", StringType(), True), StructField("CustomerID", StringType(), True),
        StructField("CustomerName", StringType(), True), StructField("Profit", DoubleType(), True)
    ])

    # Customer 2 yields higher profit than Customer 1. The output must sort Customer 2 first.
    mock_data = [
        Row(OrderDate="20/06/2026", Category="Tech", SubCategory="App", CustomerID="CUST_01", CustomerName="LowEarner", Profit=10.00),
        Row(OrderDate="20/06/2026", Category="Tech", SubCategory="App", CustomerID="CUST_02", CustomerName="HighEarner", Profit=5000.00)
    ]
    
    spark.createDataFrame(mock_data, schema=schema).createOrReplaceTempView("v_cust_orders")

    # Act
    result_df = create_gold_profit_aggregation("v_cust_orders")
    results = result_df.collect()

    # Assert
    assert result_df.count() == 2
    
    # Verify Sorting Rule Contract (Index 0 MUST be the high earner)
    assert results[0]["CustomerID"] == "CUST_02", "Sorting failed! Highest profit wasn't returned first."
    assert results[0]["TotalProfit"] == 5000.00
    assert results[1]["CustomerID"] == "CUST_01"

In [0]:
test_aggregation_by_customer_and_ranking_order(spark)